# Projet AAA : ce qui n'a pas marché

Pol BAILLEUL et Martin LEVALOIS, IMT Atlantique, 2026-2027

Ce notebook complète le notebook principal (`Notebook_Bailleul_Levalois_v2.ipynb`). On y garde les essais qui n'ont pas marché, avec ce qu'on en a retenu.

## Essai 1 : découpage train / test avec `StratifiedGroupKFold`
Pour éviter que des instances identiques soient à la fois dans le train et le test, on voulait un découpage groupé (un groupe = marque + description) et stratifié sur l'étiquette énergie. On a d'abord essayé `StratifiedGroupKFold` (5 plis, on prend un pli comme test).


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42


pandas 2.2.3 | scikit-learn 1.6.1


In [ ]:
# Copie du fichier ADEME hébergée sur le GitHub de l'équipe
GITHUB_RAW_URL = "https://raw.githubusercontent.com/Polbailleul/AAA-Project-Levalois-Bailleul/main/ademe_vehicules_co2.csv"

# Le fichier est encodé en latin-1 (le UTF-8 échoue) et séparé par des ";"
df_raw = pd.read_csv(GITHUB_RAW_URL, sep=";", encoding="latin-1", low_memory=False)
print(f"Fichier brut : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes")
display(df_raw.head(3))


Fichier brut : 55044 lignes x 30 colonnes


,lib_mrq,lib_mod_doss,lib_mod,dscom,cnit,tvv,cod_cbr,hybride,puiss_admin_98,puiss_max,...,masse_ordma_min,masse_ordma_max,champ_v9,date_maj,Carrosserie,gamme,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,ALFA-ROMEO,159,159,159 1750 Tbi (200ch),M10ALFVP000G340,939AXN1B52C,ES,non,12,147,...,1505,1505,715/2007*692/2008EURO5,mars-14,BERLINE,MOY-SUPER,NaN,NaN,NaN,NaN
1,ALFA-ROMEO,159,159,159 1750 Tbi (200ch),M10ALFVP000H341,939BXN1B53C,ES,non,12,147,...,1555,1555,715/2007*692/2008EURO5,mars-14,BERLINE,MOY-SUPER,NaN,NaN,NaN,NaN
2,ALFA-ROMEO,159,159,159 2.0 JTDm (136ch),M10ALFVP000E302,939AXR1B64,GO,non,7,100,...,1565,1565,715/2007*692/2008EURO5,mars-14,BERLINE,MOY-SUPER,NaN,NaN,NaN,NaN


In [ ]:
# Noms de colonnes explicites
RENAME = {
    "lib_mrq": "marque", "lib_mod": "modele", "dscom": "description",
    "cod_cbr": "carburant", "hybride": "hybride",
    "puiss_admin_98": "puiss_admin_cv", "puiss_max": "puiss_max_kw",
    "typ_boite_nb_rapp": "boite_rapports",
    "conso_urb": "conso_urb", "conso_exurb": "conso_exurb", "conso_mixte": "conso_mixte",
    "co2": "co2_g_km", "co_typ_1": "co_g_km", "hc": "hc_g_km", "nox": "nox_g_km",
    "hcnox": "hcnox_g_km", "ptcl": "particules_g_km",
    "masse_ordma_min": "masse_min_kg", "masse_ordma_max": "masse_max_kg",
    "champ_v9": "norme_euro_brut", "Carrosserie": "carrosserie", "gamme": "gamme",
}
# Colonnes écartées et la raison
DROPPED = {
    "lib_mod_doss": "nom de dossier du modèle, quasi identique à lib_mod",
    "cnit": "identifiant unique de chaque version (aucune information mécanique)",
    "tvv": "code type-variante-version (identifiant)",
    "date_maj": "date de mise à jour, presque toujours vide",
}
NUM_COLS = ["puiss_admin_cv", "puiss_max_kw", "masse_min_kg", "masse_max_kg",
            "conso_urb", "conso_exurb", "conso_mixte",
            "co_g_km", "hc_g_km", "nox_g_km", "hcnox_g_km", "particules_g_km", "co2_g_km"]
BINS = [0, 100, 120, 140, 160, 200, 250, np.inf]
LABELS = ["A", "B", "C", "D", "E", "F", "G"]


def clean_ademe(raw):
    """Nettoyage du fichier brut. Retourne le tableau propre et le journal des lignes."""
    log = [("Fichier brut", len(raw))]
    d = raw.loc[:, ~raw.columns.str.startswith("Unnamed")].copy()   # colonnes vides créées par les ";" finaux
    d = d.drop_duplicates()
    log.append(("Après suppression des doublons exacts", len(d)))
    d = d.drop(columns=list(DROPPED)).rename(columns=RENAME)

    for c in NUM_COLS:                                              # virgule par point
        d[c] = pd.to_numeric(d[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

    # Norme Euro : on garde EURO5, EURO6 ou EEV et le reste (texte tronqué) devient NaN
    d["norme_euro"] = (d["norme_euro_brut"].astype(str)
                       .str.extract(r"(EURO\s?[0-9]|EEV)", expand=False).str.replace(" ", "", regex=False))
    d = d.drop(columns="norme_euro_brut")

    d = d[d["co2_g_km"].notna() & (d["co2_g_km"] > 0)]              # sans CO2 connu, ligne inutilisable
    log.append(("Après retrait des lignes sans CO2 valide", len(d)))

    # Variables cibles dérivées du CO2
    d["etiquette_energie"] = pd.cut(d["co2_g_km"], bins=BINS, labels=LABELS)
    d["malus_eleve"] = (d["co2_g_km"] > 160).astype(int)
    # Identifiant de groupe : une même description commerciale = un groupe (sert au découpage plus loin)
    d["groupe"] = d["marque"].astype(str) + " | " + d["description"].astype(str)
    return d.reset_index(drop=True), pd.DataFrame(log, columns=["Étape", "Lignes restantes"])


df, cleaning_log = clean_ademe(df_raw)
display(cleaning_log)
print("\nColonnes écartées :")
display(pd.Series(DROPPED, name="Raison").to_frame())


,Étape,Lignes restantes
0,Fichier brut,55044
1,Après suppression des doublons exacts,55037
2,Après retrait des lignes sans CO2 valide,55003



Colonnes écartées :


,Raison
lib_mod_doss,"nom de dossier du modèle, quasi identique à li..."
cnit,identifiant unique de chaque version (aucune i...
tvv,code type-variante-version (identifiant)
date_maj,"date de mise à jour, presque toujours vide"


In [ ]:
# Les 5 plis possibles : taille du test et écart entre la répartition des étiquettes du test et celle de la base
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
ref = df["etiquette_energie"].value_counts(normalize=True).reindex(LABELS)
lignes = []
for k, (tr, te) in enumerate(sgkf.split(df, df["etiquette_energie"], groups=df["groupe"]), 1):
    rep = df.iloc[te]["etiquette_energie"].value_counts(normalize=True).reindex(LABELS)
    lignes.append({"pli": k, "% test": round(len(te) / len(df) * 100, 1),
                   "écart étiquettes (pts)": round((rep - ref).abs().sum() * 100, 1),
                   "% de F dans le test": round(rep["F"] * 100, 1)})
display(pd.DataFrame(lignes).set_index("pli"))


,% test,écart étiquettes (pts),% de F dans le test
pli,,,
1,13.8,17.3,60.7
2,28.5,5.9,57.1
3,20.5,40.0,74.5
4,27.1,30.3,41.8
5,10.2,41.4,36.3


Les plis sont très déséquilibrés : le test fait entre 10 % et 28 % des instances selon le pli, et la répartition des classes peut s'écarter de plus de 40 points de celle de la base. Sur un autre ordinateur avec scikit-learn 1.9, on obtenait au contraire des plis réguliers (autour de 20 %). Le résultat dépend donc de la version utilisée, ce qui ne convient pas pour un découpage qu'on veut reproductible.

Le problème vient sans doute des groupes énormes (jusqu'à 5 874 instances pour une seule description de Mercedes) : il est difficile de faire des plis à la fois groupés, stratifiés et de même taille.

Dans le notebook principal, on a donc remplacé cette méthode par `GroupShuffleSplit` avec 100 tirages, en gardant celui dont la taille et la répartition des classes sont les plus proches de la base. Ce découpage donne le même résultat avec les deux versions de scikit-learn.
